# Operator Wrap-Up

PKS의 `burgers1d`, `darcy2d` 결과를 정리하고, 비어 있는 `navierstokes` 폴더는 unavailable 상태로 명시합니다.


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() and (candidate / "notebook").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOK_ROOT = REPO_ROOT / "notebook" / "FINAL_WRAPUP"
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from _shared.io import load_curve_file, load_epoch_metrics, load_pks_results, load_run_tree
from _shared.plotting import (
    DATASET_ORDER,
    METHOD_COLORS,
    METHOD_ORDER,
    apply_report_style,
    pretty_dataset,
    pretty_method,
    pretty_model,
    pretty_pair,
    save_figure,
    save_table,
)

apply_report_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

NOTEBOOK_ROOT


## Load


In [ ]:
PKS_ROOT = REPO_ROOT / "notebook" / "results_from_pks" / "result_zip"
operator = load_pks_results(PKS_ROOT)
operator = operator[operator["task"] == "operator"].copy()
operator_file_count = operator["source_path"].nunique()
if operator_file_count != 18:
    raise AssertionError(f"Expected 18 operator PKS CSV files, found {operator_file_count}")

navierstokes_dir = PKS_ROOT / "navierstokes"
navierstokes_csv = sorted(navierstokes_dir.glob("*.csv")) if navierstokes_dir.exists() else []
if navierstokes_csv:
    raise AssertionError("Expected the navierstokes PKS folder to be empty for this wrap-up.")

display(Markdown("**Data unavailable:** `navierstokes` currently has no PKS CSV artifacts, so this notebook reports only Burgers-1D and Darcy-2D."))
display(operator.head())


## Normalize


In [ ]:
def stack_curves(curves):
    prepared = []
    for curve in curves:
        if curve is None:
            continue
        arr = np.asarray(curve, dtype=float).reshape(-1)
        if arr.size:
            prepared.append(arr)
    if not prepared:
        raise ValueError("No non-empty curves were provided.")
    min_len = min(len(arr) for arr in prepared)
    return np.vstack([arr[:min_len] for arr in prepared])

def mean_and_std(curves):
    stacked = stack_curves(curves)
    return stacked.mean(axis=0), stacked.std(axis=0)


operator["dataset_label"] = operator["dataset"].map(pretty_dataset)
operator["model_label"] = operator["model"].map(pretty_model)
operator["peer_label"] = operator["peer_model"].map(pretty_model)

def config_label(row):
    if row["mode"] == "single_baseline":
        return f"{pretty_model(row['model'])} (single)"
    return f"{pretty_model(row['model'])} | peer={pretty_model(row['peer_model'])}"

operator["config_label"] = operator.apply(config_label, axis=1)

operator_summary = (
    operator.groupby(["dataset", "dataset_label", "config_label"], dropna=False)
    .agg(
        mean_error=("metric_value", "mean"),
        std_error=("metric_value", "std"),
        n_seeds=("seed", "nunique"),
    )
    .reset_index()
    .sort_values(["dataset", "config_label"])
)

display(operator_summary)


## Summary Table


In [ ]:
summary_export_path = save_table(operator_summary, "operator", "operator_summary")
display(Markdown(f"Saved summary table to `{summary_export_path}`."))
operator_summary


## Main Figures


In [ ]:
datasets = sorted(operator_summary["dataset"].unique(), key=lambda x: DATASET_ORDER.get(x, 99))
fig, axes = plt.subplots(1, len(datasets), figsize=(16, 5), sharey=False)
axes = np.atleast_1d(axes)

for ax, dataset in zip(axes, datasets):
    ds = operator_summary[operator_summary["dataset"] == dataset].copy().reset_index(drop=True)
    x = np.arange(len(ds))
    ax.bar(x, ds["mean_error"], color="#bf616a")
    std_values = ds["std_error"].fillna(0.0).to_numpy()
    ax.errorbar(x, ds["mean_error"], yerr=std_values, fmt="none", ecolor="#2e3440", capsize=4)
    ax.set_title(pretty_dataset(dataset))
    ax.set_ylabel("Best error")
    ax.set_xticks(x)
    ax.set_xticklabels(ds["config_label"], rotation=45, ha="right")

fig.suptitle("Dataset-wise Best Error", fontsize=16)
fig.tight_layout()
main_path = save_figure(fig, "operator", "operator_best_error")
display(Markdown(f"Saved main figure to `{main_path}`."))
plt.show()


## Secondary Figures


In [ ]:
datasets = sorted(operator["dataset"].unique(), key=lambda x: DATASET_ORDER.get(x, 99))
fig, axes = plt.subplots(len(datasets), 2, figsize=(16, 10), sharex=False)
axes = np.atleast_2d(axes)

for row_idx, dataset in enumerate(datasets):
    ds = operator[operator["dataset"] == dataset].copy()
    train_ax = axes[row_idx, 0]
    test_ax = axes[row_idx, 1]

    for config_label, group in ds.groupby("config_label"):
        train_mean, train_std = mean_and_std(group["train_curve"])
        test_mean, test_std = mean_and_std(group["test_curve"])
        epochs = np.arange(1, len(train_mean) + 1)

        train_ax.plot(epochs, train_mean, linewidth=2.0, label=config_label)
        train_ax.fill_between(epochs, train_mean - train_std, train_mean + train_std, alpha=0.15)

        test_ax.plot(epochs, test_mean, linewidth=2.0, label=config_label)
        test_ax.fill_between(epochs, test_mean - test_std, test_mean + test_std, alpha=0.15)

    train_ax.set_title(f"{pretty_dataset(dataset)} | Train")
    train_ax.set_ylabel("Train error")
    test_ax.set_title(f"{pretty_dataset(dataset)} | Test")
    test_ax.set_ylabel("Test error")

axes[-1, 0].set_xlabel("Epoch")
axes[-1, 1].set_xlabel("Epoch")
axes[0, 1].legend(fontsize=8, ncol=2)
fig.suptitle("Representative Train/Test Curves", fontsize=16)
fig.tight_layout()
curve_path = save_figure(fig, "operator", "operator_train_test_curves")
display(Markdown(f"Saved curve figure to `{curve_path}`."))
plt.show()


In [ ]:
datasets = sorted(operator["dataset"].unique(), key=lambda x: DATASET_ORDER.get(x, 99))
fig, axes = plt.subplots(1, len(datasets), figsize=(16, 5), sharey=False)
axes = np.atleast_1d(axes)

for ax, dataset in zip(axes, datasets):
    ds = operator[operator["dataset"] == dataset].copy()
    labels = sorted(ds["config_label"].unique())
    positions = np.arange(len(labels))
    for pos, label in enumerate(labels):
        values = ds.loc[ds["config_label"] == label, "metric_value"].to_numpy()
        ax.scatter(np.full_like(values, pos, dtype=float), values, s=45, alpha=0.8)

    ax.set_title(pretty_dataset(dataset))
    ax.set_ylabel("Best error")
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=45, ha="right")

fig.suptitle("Seed-Level Dispersion", fontsize=16)
fig.tight_layout()
dispersion_path = save_figure(fig, "operator", "operator_seed_dispersion")
display(Markdown(f"Saved dispersion figure to `{dispersion_path}`."))
plt.show()


## Export


In [ ]:
export_frame = operator.drop(columns=["train_curve", "test_curve"]).copy()
export_path = save_table(export_frame, "operator", "operator_runs")
display(Markdown(f"Saved normalized run table to `{export_path}`."))


## Notes

- operator metric은 lower-is-better error로 해석합니다.
- `navierstokes`는 현재 PKS 결과가 없으므로 unavailable로 명시하고 억지로 채우지 않습니다.
- pair 결과는 각 model 관점을 분리해 같은 테이블에서 비교합니다.
